In [ ]:
#%pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [1]:
import os
from crewai import LLM

# Groq llm client
# llm = LLM(model="groq/openai/gpt-oss-20b")

# docker llm client
# llm = LLM(model="openai/ai/llama3.2:1B-Q8_0", base_url="http://localhost:12434/engines/v1")

# LLM setup using litellm
llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")

# LLM setup without litellm
# llm = LLM(model="openai/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434/v1")

In [2]:
llm.call("Hello")

'Hello. Is there something I can help you with or would you like to talk about something in particular?'

In [3]:
from crewai.tools import tool
from crewai_tools import WebsiteSearchTool

# Instantiate Web Search Tool
@tool
def web_search_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = WebsiteSearchTool(
        website='https://example.com',
        config=dict(
            llm=dict(
                provider="ollama", # or google, openai, anthropic, llama2, ...
                config=dict(
                    model="llama3.2:1b-instruct-q8_0",
                    # temperature=0.5,
                    # top_p=1,
                    # stream=true,
                ),
            ),
            embedder=dict(
                provider="ollama", # or openai, ollama, ...
                config=dict(
                    model_name="all-minilm",
                    task_type="RETRIEVAL_DOCUMENT",
                    # title="Embeddings",
                ),
            ),
        )
    )

    return search_tool.run(query)


##### Create Agents

In [4]:
from crewai import Agent

#Create Agents
researcher = Agent(
    role='Market Research Analyst',
    goal='Provide up-to-date market analysis of the AI industry',
    backstory='An expert analyst with a keen eye for market trends.',
    tools=[web_search_tool],  # Only uses web search tool
    verbose=True,
    llm=llm,
    allow_delegation=False,
)

writer = Agent(
    role='Content Writer',
    goal='Craft engaging blog posts about the AI industry',
    backstory='A skilled writer with a passion for technology.',
    tools=[],  # No tools needed; writes based on research summary
    verbose=True,
    llm=llm,
    allow_delegation=False,
)

##### Create Tasks

In [5]:

from crewai import Task

# Define Tasks
research = Task(
    description='Search the web for the latest AI trends and provide a summarized report.',
    expected_output='A summary of the top 3 trending developments in AI with insights on their impact.',
    agent=researcher
)

write = Task(
    description='Write an engaging blog post about the AI industry based on the research analyst’s summary.',
    expected_output='A well-structured, 4-paragraph blog post in markdown format with simple, engaging content.',
    agent=writer,
    output_file='agentic-ai/crewai/blog-posts/new_post.md',  # Saves blog post in 'blog-posts' directory
)

##### Create Crew

In [6]:

# Assemble the Crew
from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer],
    tasks=[research, write],
    #process=Process.hierarchical,
    manager_llm=llm,
    planning=True,  # Enable AI planning feature
    planning_llm=llm,
    full_output=True,
    share_crew=False,
    verbose=True
)

# Execute the Tasks
crew.kickoff()


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 17de9434-2b0a-4547-bc90-1e36c7fbec38                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-04-15 18:07:11][INFO]: Planning the crew execution


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Search the web for the latest AI trends and provide a summarized report.       │
│                  "task_description": Search the web for the latest AI trends and provide a summarized report.   │
│                  "task_expected_output": A summary of the top 3 trending developments in AI with insights on    │
│  their impact.                                                                                                  │
│                  "agent": Market Research Analyst                                                               │
│                  "agent_goal": Provide up-to-date market analysis of the AI industry                            │
│                  "task_tools": [Tool(name='web_search_tool', description='Tool Name: web_search_tool\nTool      │
│  Arguments: {\n  "properties": {\n    "query": {\n      "title": "Query",\n      "type": "string"\n    }\n      │
│  },\n  "required": [\n    "query"\n  ],\n  "title": "Web_Search_Tool",\n  "type": "object",\n                   │
│  "additionalProperties": false\n}\nTool Description: \n    Searches the web and returns results.\n    ',        │
│  env_vars=[], args_schema=<class 'crewai.tools.base_tool.Web_Search_Tool'>, description_updated=False,          │
│  cache_function=<function _default_cache_function at 0x765c62418b80>, result_as_answer=False,                   │
│  max_usage_count=None, current_usage_count=0, func=<function web_search_tool at 0x765c5243e0c0>,                │
│  tool_type='crewai.tools.base_tool.Tool')]                                                                      │
│                  "agent_tools": [name='web_search_tool' description='Tool Name: web_search_tool\nTool           │
│  Arguments: {\n  "properties": {\n    "query": {\n      "title": "Query",\n      "type": "string"\n    }\n      │
│  },\n  "required": [\n    "query"\n  ],\n  "title": "Web_Search_Tool",\n  "type": "object",\n                   │
│  "additionalProperties": false\n}\nTool Description: \n    Searches the web and returns results.\n    '         │
│  env_vars=[] args_schema=<class 'crewai.tools.base_tool.Web_Search_Tool'> description_updated=False             │
│  cache_function=<function _default_cache_function at 0x765c62418b80> result_as_answer=False                     │
│  max_usage_count=None current_usage_count=0 func=<function web_search_tool at 0x765c5243e0c0>                   │
│  tool_type='crewai.tools.base_tool.Tool']                                                                       │
│                  Task Number 2 - Write an engaging blog post about the AI industry based on the research        │
│  analyst’s summary.                                                                                             │
│                  "task_description": Write an engaging blog post about the AI industry based on the research    │
│  analyst’s summary.                                                                                             │
│                  "task_expected_output": A well-structured, 4-paragraph blog post in markdown format with       │
│  simple, engaging content.                                                                                      │
│                  "agent": Content Writer                                                                        │
│                  "agent_goal": Craft engaging blog posts about the AI industry                                  │
│                  "task_tools": []                      

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Search the web for the latest AI trends and provide a summarized report.       │
│                  "task_description": Search the web for the latest AI trends and provide a summarized report.   │
│                  "task_expected_output": A summary of the top 3 trending developments in AI with insights on    │
│  their impact.                                                                                                  │
│                  "agent": Market Research Analyst                                                               │
│                  "agent_goal": Provide up-to-date market analysis of the AI industry                            │
│                  "task_tools": [Tool(name='web_search_tool', description='Tool Name: web_search_tool\nTool      │
│  Arguments: {\n  "properties": {\n    "query": {\n      "title": "Query",\n      "type": "string"\n    }\n      │
│  },\n  "required": [\n    "query"\n  ],\n  "title": "Web_Search_Tool",\n  "type": "object",\n                   │
│  "additionalProperties": false\n}\nTool Description: \n    Searches the web and returns results.\n    ',        │
│  env_vars=[], args_schema=<class 'crewai.tools.base_tool.Web_Search_Tool'>, description_updated=False,          │
│  cache_function=<function _default_cache_function at 0x765c62418b80>, result_as_answer=False,                   │
│  max_usage_count=None, current_usage_count=0, func=<function web_search_tool at 0x765c5243e0c0>,                │
│  tool_type='crewai.tools.base_tool.Tool')]                                                                      │
│                  "agent_tools": [name='web_search_tool' description='Tool Name: web_search_tool\nTool           │
│  Arguments: {\n  "properties": {\n    "query": {\n      "title": "Query",\n      "type": "string"\n    }\n      │
│  },\n  "required": [\n    "query"\n  ],\n  "title": "Web_Search_Tool",\n  "type": "object",\n                   │
│  "additionalProperties": false\n}\nTool Description: \n    Searches the web and returns results.\n    '         │
│  env_vars=[] args_schema=<class 'crewai.tools.base_tool.Web_Search_Tool'> description_updated=False             │
│  cache_function=<function _default_cache_function at 0x765c62418b80> result_as_answer=False                     │
│  max_usage_count=None current_usage_count=0 func=<function web_search_tool at 0x765c5243e0c0>                   │
│  tool_type='crewai.tools.base_tool.Tool']                                                                       │
│                  Task Number 2 - Write an engaging blog post about the AI industry based on the research        │
│  analyst’s summary.                                                                                             │
│                  "task_description": Write an engaging blog post about the AI industry based on the research    │
│  analyst’s summary.                                                                                             │
│                  "task_expected_output": A well-structured, 4-paragraph blog post in markdown format with       │
│  simple, engaging content.                                                                                      │
│                  "agent": Content Writer                                                                        │
│                  "agent_goal": Craft engaging blog posts about the AI industry                                  │
│                  "task_tools": []                      

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the web for the latest AI trends and provide a summarized report.Execute Task Number 1 to search  │
│  the web, use CrewAI's Web Search Tool or an alternative browser extension like Google Trend or TinEye to find  │
│  relevant articles. Input keywords like 'latest AI trends', 'deep learning advancements' and submit your        │
│  search query for results. Analyze the top 3 trending developments in AI based on the report, incorporating     │
│  insights from reputable sources such as IBM Watson, Microsoft AI, and Carnegie Mellon University.              │
│  ID: 394b850f-829c-4560-8b4f-37b25460b869                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task: Write an engaging blog post about the AI industry based on the research analyst’s summary.Create Task    │
│  Number 1 to search for articles or blog posts that discuss the latest trends in the field. Utilize an online   │
│  platform like Medium, LinkedIn Pulse, or a relevant publication to gather content. Then, write an engaging     │
│  and informative blog post using tools available such as Grammarly, Hemingway Editor, or a simple               │
│  text-to-image converter in Visual Studio Code for visualizing AI-related images.                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### The AI Revolution: Trends, Advancements, and Innovations                                                   │
│                                                                                                                 │
│  #### Introduction                                                                                              │
│  The world of artificial intelligence (AI) is rapidly evolving at an incredible pace. With the rapid progress   │
│  in technologies like machine learning, natural language processing, computer vision, and neural networks, AI   │
│  has become an indispensable part of our daily lives. From virtual assistants to self-driving cars, AI is       │
│  transforming various industries and revolutionizing the way we interact with technology.                       │
│                                                                                                                 │
│  #### Latest Trends in AI Research                                                                              │
│  According to research from renowned organizations and institutions, some of the most exciting trends in AI     │
│  include:                                                                                                       │
│                                                                                                                 │
│  *   **Advancements in Deep Learning**: Deep learning algorithms have achieved state-of-the-art results in      │
│  tasks like image classification, speech recognition, and natural language processing. This has led to          │
│  breakthroughs in areas such as computer vision, robotics, and human-computer interaction.                      │
│  *   **Increased Adoption of Edge Computing**: With the growing demand for AI-driven applications on the go,    │
│  edge computing is becoming an increasingly important aspect of the industry. Edge computing enables real-time  │
│  processing and computation closer to where data is produced, reducing latency and improving overall            │
│  efficiency.                                                                                                    │
│  *   **Growing Focus on Natural Language Processing (NLP)**: NLP has seen significant advancements in recent    │
│  years, with improvements in areas such as language translation, text summarization, and sentiment analysis.    │
│                                                                                                                 │
│  #### The Role of AI in Society                                                                                 │
│  As AI technology continues to advance at an unprecedented rate, its impact on society will only grow. Areas    │
│  like healthcare, education, and the justice system are already seeing significant improvements through         │
│  AI-powered solutions that improve accessibility, accuracy, and fairness.                                       │
│                                                                                                                 │
│  #### Challenges and Opportunities                                                                              │
│                                                                                                                 │
│  *   **Addressing Biases**: With AI playing a crucial r

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write an engaging blog post about the AI industry based on the research analyst’s summary.Create Task    │
│  Number 1 to search for articles or blog posts that discuss the latest trends in the field. Utilize an online   │
│  platform like Medium, LinkedIn Pulse, or a relevant publication to gather content. Then, write an engaging     │
│  and informative blog post using tools available such as Grammarly, Hemingway Editor, or a simple               │
│  text-to-image converter in Visual Studio Code for visualizing AI-related images.                               │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 17de9434-2b0a-4547-bc90-1e36c7fbec38                                                                       │
│  Final Output: ### The AI Revolution: Trends, Advancements, and Innovations                                     │
│                                                                                                                 │
│  #### Introduction                                                                                              │
│  The world of artificial intelligence (AI) is rapidly evolving at an incredible pace. With the rapid progress   │
│  in technologies like machine learning, natural language processing, computer vision, and neural networks, AI   │
│  has become an indispensable part of our daily lives. From virtual assistants to self-driving cars, AI is       │
│  transforming various industries and revolutionizing the way we interact with technology.                       │
│                                                                                                                 │
│  #### Latest Trends in AI Research                                                                              │
│  According to research from renowned organizations and institutions, some of the most exciting trends in AI     │
│  include:                                                                                                       │
│                                                                                                                 │
│  *   **Advancements in Deep Learning**: Deep learning algorithms have achieved state-of-the-art results in      │
│  tasks like image classification, speech recognition, and natural language processing. This has led to          │
│  breakthroughs in areas such as computer vision, robotics, and human-computer interaction.                      │
│  *   **Increased Adoption of Edge Computing**: With the growing demand for AI-driven applications on the go,    │
│  edge computing is becoming an increasingly important aspect of the industry. Edge computing enables real-time  │
│  processing and computation closer to where data is produced, reducing latency and improving overall            │
│  efficiency.                                                                                                    │
│  *   **Growing Focus on Natural Language Processing (NLP)**: NLP has seen significant advancements in recent    │
│  years, with improvements in areas such as language translation, text summarization, and sentiment analysis.    │
│                                                                                                                 │
│  #### The Role of AI in Society                                                                                 │
│  As AI technology continues to advance at an unprecedented rate, its impact on society will only grow. Areas    │
│  like healthcare, education, and the justice system are already seeing significant improvements through         │
│  AI-powered solutions that improve accessibility, accuracy, and fairness.                                       │
│                                                                                                                 │
│  #### Challenges and Opportunities                                                                              │
│                                                                                                                 │
│  *   **Addressing Biases**: With AI playing a crucial 

CrewOutput(raw='### The AI Revolution: Trends, Advancements, and Innovations\n\n#### Introduction\nThe world of artificial intelligence (AI) is rapidly evolving at an incredible pace. With the rapid progress in technologies like machine learning, natural language processing, computer vision, and neural networks, AI has become an indispensable part of our daily lives. From virtual assistants to self-driving cars, AI is transforming various industries and revolutionizing the way we interact with technology.\n\n#### Latest Trends in AI Research\nAccording to research from renowned organizations and institutions, some of the most exciting trends in AI include:\n\n*   **Advancements in Deep Learning**: Deep learning algorithms have achieved state-of-the-art results in tasks like image classification, speech recognition, and natural language processing. This has led to breakthroughs in areas such as computer vision, robotics, and human-computer interaction.\n*   **Increased Adoption of Edge 